# Modern CNNs with PyTorch

**Module 04: CNN**  
**Objective**: Master modern CNN architectures and techniques

Modern CNNs go beyond simple stacking:
- **ResNet**: Skip connections solve vanishing gradients
- **BatchNorm**: Stabilizes training
- **Transfer Learning**: Use pretrained models
- **Data Augmentation**: Improve generalization

## What You'll Learn
1. Building ResNet from scratch
2. Transfer learning with pretrained models
3. Fine-tuning strategies
4. Advanced augmentation techniques
5. Feature visualization
6. Class Activation Maps (CAM)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

## 1. ResNet: Residual Networks

**Problem**: Deep networks suffer from vanishing gradients → training degrades

**Solution**: Skip connections allow gradients to flow directly

$$\mathbf{y} = \mathcal{F}(\mathbf{x}, \{W_i\}) + \mathbf{x}$$

**Key insight**: Easier to learn residual $\mathcal{F}(\mathbf{x})$ than full mapping $\mathcal{H}(\mathbf{x})$

In [ ]:
class ResidualBlock(nn.Module):
    """Basic residual block."""
    
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Skip connection (identity or projection)
        self.skip = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        identity = self.skip(x)
        
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        out += identity  # Skip connection
        out = F.relu(out)
        
        return out

class ResNet(nn.Module):
    """ResNet architecture."""
    
    def __init__(self, num_blocks, num_classes=10):
        super().__init__()
        
        self.in_channels = 64
        
        # Initial convolution
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        
        # Residual layers
        self.layer1 = self._make_layer(64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(512, num_blocks[3], stride=2)
        
        # Classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
    
    def _make_layer(self, out_channels, num_blocks, stride):
        layers = []
        
        # First block (may downsample)
        layers.append(ResidualBlock(self.in_channels, out_channels, stride))
        self.in_channels = out_channels
        
        # Remaining blocks
        for _ in range(num_blocks - 1):
            layers.append(ResidualBlock(out_channels, out_channels, stride=1))
        
        return nn.Sequential(*layers)
    
    def forward(self, x):
        # Initial conv
        out = F.relu(self.bn1(self.conv1(x)))
        
        # Residual layers
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        
        # Classifier
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        
        return out

def ResNet18(num_classes=10):
    return ResNet([2, 2, 2, 2], num_classes)

def ResNet34(num_classes=10):
    return ResNet([3, 4, 6, 3], num_classes)

# Test ResNet
model = ResNet18(num_classes=10).to(device)
x = torch.randn(2, 3, 32, 32).to(device)
y = model(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {y.shape}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}\n")

# Count layers
total_layers = sum(1 for _ in model.modules() if isinstance(_, (nn.Conv2d, nn.Linear)))
print(f"Total Conv+FC layers: {total_layers}")

## 2. Training on CIFAR-10

In [ ]:
# CIFAR-10 dataset
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, 
                                              download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                             download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}\n")

# Visualize samples
def show_images(loader, n=8):
    images, labels = next(iter(loader))
    
    fig, axes = plt.subplots(1, n, figsize=(15, 2))
    for i in range(n):
        img = images[i].numpy().transpose((1, 2, 0))
        img = img * np.array([0.2023, 0.1994, 0.2010]) + np.array([0.4914, 0.4822, 0.4465])
        img = np.clip(img, 0, 1)
        
        axes[i].imshow(img)
        axes[i].set_title(classes[labels[i]])
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

show_images(train_loader)

In [ ]:
def train_model(model, train_loader, test_loader, epochs=10, lr=0.1):
    """Training loop with learning rate scheduling."""
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    train_losses, test_losses = [], []
    train_accs, test_accs = [], []
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for inputs, targets in pbar:
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
            pbar.set_postfix({'loss': train_loss/(pbar.n+1), 'acc': 100.*correct/total})
        
        train_losses.append(train_loss / len(train_loader))
        train_accs.append(100. * correct / total)
        
        # Testing
        model.eval()
        test_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                
                test_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()
        
        test_losses.append(test_loss / len(test_loader))
        test_accs.append(100. * correct / total)
        
        print(f"Test Loss: {test_losses[-1]:.4f}, Test Acc: {test_accs[-1]:.2f}%\n")
        
        scheduler.step()
    
    return train_losses, test_losses, train_accs, test_accs

# Train ResNet-18
print("\nTraining ResNet-18 on CIFAR-10\n")
model = ResNet18(num_classes=10).to(device)

train_losses, test_losses, train_accs, test_accs = train_model(
    model, train_loader, test_loader, epochs=5, lr=0.1
)

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(train_losses, label='Train')
ax1.plot(test_losses, label='Test')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Test Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(train_accs, label='Train')
ax2.plot(test_accs, label='Test')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Test Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Test Accuracy: {test_accs[-1]:.2f}%")

## 3. Transfer Learning

**Idea**: Use pretrained models on ImageNet as feature extractors

**Two approaches**:
1. **Feature extraction**: Freeze pretrained layers, train only classifier
2. **Fine-tuning**: Unfreeze some layers, train with small learning rate

In [ ]:
def load_pretrained_resnet(num_classes=10, freeze_features=True):
    """Load pretrained ResNet and modify for CIFAR-10."""
    
    # Load pretrained ResNet-18
    model = models.resnet18(pretrained=True)
    
    # Freeze feature extractor
    if freeze_features:
        for param in model.parameters():
            param.requires_grad = False
    
    # Replace classifier
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    
    return model.to(device)

# Feature extraction
print("\nTransfer Learning: Feature Extraction\n")
model_transfer = load_pretrained_resnet(num_classes=10, freeze_features=True)

# Count trainable parameters
total_params = sum(p.numel() for p in model_transfer.parameters())
trainable_params = sum(p.numel() for p in model_transfer.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}\n")

# Train only classifier (faster, 2 epochs)
train_losses_transfer, test_losses_transfer, train_accs_transfer, test_accs_transfer = train_model(
    model_transfer, train_loader, test_loader, epochs=2, lr=0.001
)

print(f"\nTransfer Learning Test Accuracy: {test_accs_transfer[-1]:.2f}%")

## 4. Fine-Tuning

**Strategy**: 
1. Train classifier with frozen features
2. Unfreeze last layers
3. Train whole network with small LR

In [ ]:
def fine_tune_model(model, train_loader, test_loader, epochs=5, lr=0.0001):
    """Fine-tune pretrained model."""
    
    # Unfreeze all layers
    for param in model.parameters():
        param.requires_grad = True
    
    print(f"\nFine-tuning (all layers unfrozen)...\n")
    
    return train_model(model, train_loader, test_loader, epochs=epochs, lr=lr)

# Fine-tune
train_losses_ft, test_losses_ft, train_accs_ft, test_accs_ft = fine_tune_model(
    model_transfer, train_loader, test_loader, epochs=3, lr=0.0001
)

print(f"\nFine-tuned Test Accuracy: {test_accs_ft[-1]:.2f}%")

## 5. Advanced Data Augmentation

In [ ]:
# Advanced augmentation
transform_advanced = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.33)),
])

# Visualize augmentations
def visualize_augmentations(image, transform, n=6):
    """Show multiple augmented versions."""
    
    fig, axes = plt.subplots(1, n+1, figsize=(18, 3))
    
    # Original
    axes[0].imshow(image)
    axes[0].set_title('Original')
    axes[0].axis('off')
    
    # Augmented
    for i in range(1, n+1):
        aug_img = transform(image)
        
        # Denormalize
        if isinstance(aug_img, torch.Tensor):
            img_np = aug_img.numpy().transpose((1, 2, 0))
            img_np = img_np * np.array([0.2023, 0.1994, 0.2010]) + np.array([0.4914, 0.4822, 0.4465])
            img_np = np.clip(img_np, 0, 1)
            axes[i].imshow(img_np)
        else:
            axes[i].imshow(aug_img)
        
        axes[i].set_title(f'Augmented {i}')
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

# Load sample image
from PIL import Image
sample_img = train_dataset.data[0]
sample_img = Image.fromarray(sample_img)

visualize_augmentations(sample_img, transform_advanced, n=5)

## 6. Class Activation Mapping (CAM)

**Visualize what the network "sees"**

CAM highlights discriminative regions for predictions.

In [ ]:
class GradCAM:
    """Gradient-weighted Class Activation Mapping."""
    
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks
        def forward_hook(module, input, output):
            self.activations = output
        
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]
        
        target_layer.register_forward_hook(forward_hook)
        target_layer.register_backward_hook(backward_hook)
    
    def generate_cam(self, input_image, target_class=None):
        """Generate CAM for target class."""
        
        # Forward pass
        model_output = self.model(input_image)
        
        if target_class is None:
            target_class = model_output.argmax(dim=1).item()
        
        # Backward pass
        self.model.zero_grad()
        one_hot = torch.zeros_like(model_output)
        one_hot[0, target_class] = 1
        model_output.backward(gradient=one_hot, retain_graph=True)
        
        # Compute CAM
        gradients = self.gradients.cpu().data.numpy()[0]
        activations = self.activations.cpu().data.numpy()[0]
        
        weights = np.mean(gradients, axis=(1, 2))
        cam = np.zeros(activations.shape[1:], dtype=np.float32)
        
        for i, w in enumerate(weights):
            cam += w * activations[i]
        
        cam = np.maximum(cam, 0)
        cam = cam / (cam.max() + 1e-8)
        
        return cam, target_class

def show_cam_on_image(img, cam, alpha=0.5):
    """Overlay CAM on image."""
    
    # Resize CAM to image size
    from scipy.ndimage import zoom
    cam = zoom(cam, (img.shape[0] / cam.shape[0], img.shape[1] / cam.shape[1]))
    
    # Apply colormap
    cam = plt.cm.jet(cam)[:, :, :3]
    
    # Overlay
    output = (1 - alpha) * img + alpha * cam
    output = np.clip(output, 0, 1)
    
    return output

# Visualize CAM
model.eval()
grad_cam = GradCAM(model, model.layer4[-1])  # Last residual block

# Get test image
test_image, test_label = test_dataset[10]
test_image_tensor = test_image.unsqueeze(0).to(device)

# Generate CAM
cam, pred_class = grad_cam.generate_cam(test_image_tensor)

# Denormalize image
img_np = test_image.numpy().transpose((1, 2, 0))
img_np = img_np * np.array([0.2023, 0.1994, 0.2010]) + np.array([0.4914, 0.4822, 0.4465])
img_np = np.clip(img_np, 0, 1)

# Show results
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(img_np)
axes[0].set_title(f'Original\nTrue: {classes[test_label]}')
axes[0].axis('off')

axes[1].imshow(cam, cmap='jet')
axes[1].set_title(f'CAM\nPred: {classes[pred_class]}')
axes[1].axis('off')

overlay = show_cam_on_image(img_np, cam)
axes[2].imshow(overlay)
axes[2].set_title('Overlay')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 7. Model Comparison

In [ ]:
# Compare different architectures
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

models_comparison = {
    'ResNet-18': ResNet18(num_classes=10),
    'ResNet-34': ResNet34(num_classes=10),
}

print("\nModel Comparison\n" + "="*50)
print(f"{'Model':<20} {'Parameters':>15}")
print("="*50)

for name, m in models_comparison.items():
    params = count_parameters(m)
    print(f"{name:<20} {params:>15,}")

print("="*50)

## Summary

You've mastered modern CNNs:
- ✅ ResNet with skip connections
- ✅ Training on CIFAR-10 (85%+ accuracy)
- ✅ Transfer learning (feature extraction)
- ✅ Fine-tuning pretrained models
- ✅ Advanced data augmentation
- ✅ Class Activation Mapping (visualization)

**Key insights**:
1. Skip connections enable training very deep networks (100+ layers)
2. Transfer learning accelerates training and improves generalization
3. Fine-tuning works better than feature extraction alone
4. Data augmentation is crucial for small datasets
5. CAM reveals what the network focuses on

**Architectures to explore**:
- EfficientNet (compound scaling)
- MobileNet (depthwise separable convolutions)
- Vision Transformers (ViT)

**Best Practices**:
1. Start with pretrained models (ImageNet)
2. Use data augmentation (RandomCrop, ColorJitter, RandomErasing)
3. Apply BatchNorm for stability
4. Use Cosine Annealing for LR scheduling
5. Monitor overfitting with validation set
6. Visualize activations to debug

## Exercises

1. **Deeper ResNet**: Implement ResNet-50 with bottleneck blocks
2. **Custom Dataset**: Apply transfer learning to your own image dataset
3. **EfficientNet**: Implement compound scaling (depth, width, resolution)